In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    apply_custom_style,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

### Load Pipeline

In [ ]:
device = "cuda:2" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "head_outputs")
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

### Utils

In [ ]:
def extract_projection_weights(selected_layer: int, selected_component: str, selected_head: int):
    """
    Extract Q and K weight matrices for a specific attention head.

    Args:
        selected_layer: Layer index
        selected_component: Component type ("SA" or "CA")
        selected_head: Head index

    Returns:
        tuple: (head_WQ, head_WK) as numpy arrays
    """
    component_type_mapping = {
        "SA": (0, "SelfAttention"),
        "CA": (1, "EncDecAttention"),
        "MLP": (2, "DenseReluDense"),
    }

    component_idx, component_name = component_type_mapping[selected_component]
    print(f"component_idx: {component_idx}, component_name: {component_name}")

    module = getattr(pipeline.model.model.decoder.block[selected_layer].layer[component_idx], component_name)
    # print(f"module: {module}")

    d_model = module.q.weight.shape[0]
    # print(f"d_model: {d_model}")
    head_dim = d_model // num_heads
    # print(f"head_dim: {head_dim}")
    head_start = selected_head * head_dim
    head_end = head_start + head_dim
    # print(f"head_start: {head_start}, head_end: {head_end}")

    head_WQ = module.q.weight[head_start:head_end, :].float().detach().cpu().numpy()
    # head_WQ = module.q.weight
    # print(f"head_WQ shape: {head_WQ.shape}")

    head_WK = module.k.weight[head_start:head_end, :].float().detach().cpu().numpy()
    # head_WK = module.k.weight
    # print(f"head_WK shape: {head_WK.shape}")

    return head_WQ, head_WK

In [ ]:
def _softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)


def _row_entropy(P: np.ndarray, axis: int = -1, eps: float = 1e-12) -> np.ndarray:
    return -np.sum(P * np.log(P + eps), axis=axis)


def _orth(A: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """Return orthonormal basis for col(A) via QR with rank trimming."""
    Q, R = np.linalg.qr(A)
    r = np.sum(np.abs(np.diag(R)) > eps)
    return Q[:, :r]


def _numerical_rank(s: np.ndarray, tol: float | None = None) -> int:
    if tol is None:
        tol = np.max(s) * max(len(s), 1) * np.finfo(s.dtype).eps
    return int(np.sum(s > tol))


def diagnose_attention(
    W_Q: np.ndarray,
    W_K: np.ndarray,
    H: np.ndarray | None = None,
    scale_by_sqrt_dk: bool = True,
    top_k: int = 10,
    eps: float = 1e-10,
) -> dict[str, Any]:
    """
    Diagnostics for a single attention head.
    W_Q, W_K: (d_k, d_h)
    H: (T, d_h) optional hidden states for softmax sharpness analysis
    """
    d_k, d_h = W_Q.shape
    assert W_K.shape == (d_k, d_h)
    M = W_Q.T @ W_K

    # SVD and basic norms
    U, s, Vt = np.linalg.svd(M, full_matrices=False)
    V = Vt.T
    spectral_norm = float(s[0]) if len(s) else 0.0
    frob_norm = float(np.linalg.norm(M, "fro"))
    nuclear_norm = float(np.sum(s))
    rank_num = _numerical_rank(s)
    spectral_gap = float(s[0] / s[1]) if len(s) > 1 and s[1] > eps else np.inf

    # Symmetric/skew decomposition
    Hsym = 0.5 * (M + M.T)
    Smew = 0.5 * (M - M.T)
    sym_energy = float(np.linalg.norm(Hsym, "fro") ** 2)
    skew_energy = float(np.linalg.norm(Smew, "fro") ** 2)
    skew_frac = float(skew_energy / (sym_energy + skew_energy + eps))

    # Principal angles
    Qb, Kb = _orth(W_Q.T), _orth(W_K.T)
    C = np.linalg.svd(Qb.T @ Kb, full_matrices=False)[1] if (Qb.size and Kb.size) else np.array([])
    principal_angles = np.arccos(np.clip(C, 0.0, 1.0))

    # Sequence-level analysis if H provided
    softmax_stats = None
    per_query = None
    if H is not None:
        assert H.shape[1] == d_h
        L = H @ M @ H.T
        if scale_by_sqrt_dk:
            L = L / np.sqrt(d_k)
        P = _softmax(L, axis=1)
        entropies = _row_entropy(P, axis=1)
        max_weight = np.max(P, axis=1)

        alpha = H @ U
        alignment_scores = (alpha**2) @ s / (np.linalg.norm(H @ U, axis=1) ** 2 + eps)

        x = alignment_scores - np.mean(alignment_scores)
        y = (-entropies) - np.mean(-entropies)
        corr = float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y) + eps))

        softmax_stats = {
            "entropy_mean": float(np.mean(entropies)),
            "entropy_std": float(np.std(entropies)),
            "max_weight_mean": float(np.mean(max_weight)),
            "max_weight_std": float(np.std(max_weight)),
            "alignment_mean": float(np.mean(alignment_scores)),
            "alignment_std": float(np.std(alignment_scores)),
            "corr_alignment_vs_neg_entropy": corr,
        }
        per_query = {"entropies": entropies, "max_weight": max_weight, "alignment_scores": alignment_scores}

    # Summary
    k = min(top_k, len(s))
    explained_nuclear_norm = np.cumsum(s[:k]) / (np.sum(s) + eps)
    explained_energy = np.cumsum(s[:k] ** 2) / (np.sum(s**2) + eps)

    summary = [
        f"(d_h={d_h}, d_k={d_k}) rank≈{rank_num}/{d_h} ‖M‖₂={spectral_norm:.4g} gap={spectral_gap:.3g}",
        f"Skew fraction={skew_frac:.3f}",
    ]
    if principal_angles.size:
        summary.append(
            f"Principal angles: median={np.degrees(np.median(principal_angles)):.1f}° max_cos={np.max(C):.3f}"
        )
    summary.append(f"Top-{k} σ: {np.array2string(s[:k], precision=4, suppress_small=True)}")
    summary.append(f"Explained (nuclear): {np.array2string(explained_nuclear_norm, precision=3, suppress_small=True)}")
    summary.append(f"Explained (Frobenius): {np.array2string(explained_energy, precision=3, suppress_small=True)}")
    if softmax_stats:
        summary.append(
            f"Entropy: {softmax_stats['entropy_mean']:.3f}±{softmax_stats['entropy_std']:.3f} "
            f"Max_p: {softmax_stats['max_weight_mean']:.3f}±{softmax_stats['max_weight_std']:.3f}"
        )
        summary.append(f"Alignment↔sharpness corr: {softmax_stats['corr_alignment_vs_neg_entropy']:.3f}")

    return {
        "spectral_norm": spectral_norm,
        "spectral_gap_sigma1_over_sigma2": spectral_gap,
        "frob_norm": frob_norm,
        "nuclear_norm": nuclear_norm,
        "rank_numerical": rank_num,
        "singular_values": s,
        "U": U,
        "V": V,
        "symmetric_part": Hsym,
        "skew_part": Smew,
        "skew_fraction_of_energy": skew_frac,
        "principal_angles_radians": principal_angles,
        "principal_cosines": C if principal_angles.size else np.array([]),
        "softmax_stats": softmax_stats,
        "per_query": per_query,
        "human_readable_summary": "\n".join(summary),
        "explained_energy": explained_energy,
        "explained_nuclear_norm": explained_nuclear_norm,
    }


### Analyze Singular Values and More

In [ ]:
heads_to_analyze_diffuse = [(0, "CA", 4), (0, "CA", 11), (1, "CA", 1), (7, "CA", 3), (9, "CA", 5), (10, "CA", 11)]
heads_to_analyze_sharp = [(4, "CA", 1), (7, "CA", 11), (3, "CA", 9), (6, "CA", 9), (4, "CA", 10), (3, "CA", 10)]
heads_to_analyze = heads_to_analyze_diffuse + heads_to_analyze_sharp

# make colors such that heads_to_analyze_diffuse are blue and heads_to_analyze_sharp are red
colors_diffuse = plt.get_cmap("Blues")(np.linspace(0.1, 0.9, len(heads_to_analyze_diffuse)))
colors_sharp = plt.get_cmap("Reds")(np.linspace(0.1, 0.9, len(heads_to_analyze_sharp)))
colors = np.concatenate([colors_diffuse, colors_sharp])

# colors = plt.get_cmap("RdBu")(np.linspace(0, 1, len(reports)))

In [ ]:
# First, collect all reports
reports = {}
for head_to_analyze in heads_to_analyze:
    selected_layer, selected_component, selected_head = head_to_analyze
    print("=" * 100)
    print(f"Analyzing Layer {selected_layer} Head {selected_head} and component {selected_component}")
    head_WQ, head_WK = extract_projection_weights(
        selected_layer=selected_layer, selected_component=selected_component, selected_head=selected_head
    )

    # W_Q, W_K have shape (d_k, d_h). H has shape (T, d_h) if you want softmax diagnostics.
    report = diagnose_attention(head_WQ, head_WK, H=None, scale_by_sqrt_dk=True, top_k=20)

    print(report["human_readable_summary"])

    # Store report with a descriptive key
    key = f"L{selected_layer}_H{selected_head}_{selected_component}"
    reports[key] = report

In [ ]:
report.keys()

In [ ]:
# plot the spectral_norm of each head (it is a scalar for each head)
plt.figure(figsize=(10, 5))
spectral_gaps = []
labels = []
for idx, (key, report) in enumerate(reports.items()):
    spectral_gap = report["spectral_gap_sigma1_over_sigma2"]
    spectral_gaps.append(spectral_gap)
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key)
    labels.append(pretty_key)
plt.bar(range(len(spectral_gaps)), spectral_gaps, color=colors, alpha=0.7)
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.xlabel("Head", fontweight="bold")
plt.ylabel("Spectral Gap", fontweight="bold")
plt.tight_layout()

In [ ]:
# Plot 1: Singular values spectrum
plt.figure(figsize=(6, 6))
for idx, (key, report) in enumerate(reports.items()):
    s = report["singular_values"]
    k = min(30, len(s))
    plt.plot(np.arange(1, k + 1), s[:k], marker="o", color=colors[idx], label=key, alpha=0.7)
plt.xlabel("Index r", fontweight="bold")
plt.ylabel("σ_r", fontweight="bold")
plt.title("Singular Values", fontweight="bold")
plt.legend(frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
for idx, (key, report) in enumerate[tuple[Any, Any]](reports.items()):
    explained_energy = report["explained_energy"]
    indices_considered = np.arange(1, len(explained_energy) + 1)
    # for key of format L{layer}_H{head}_{component}, replace {layer} with L, {head} with H, and ignore the rest
    # pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2 \3", key)
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key)
    plt.plot(indices_considered, explained_energy, marker="o", color=colors[idx], label=pretty_key, alpha=0.7)
plt.xlabel("Top singular values", fontweight="bold")
plt.xticks(indices_considered)
plt.title("Explained Energy (Frobenius Norm)", fontweight="bold")
plt.tight_layout()
plt.legend(frameon=True, loc="lower right")

In [ ]:
plt.figure(figsize=(5, 5))
for idx, (key, report) in enumerate[tuple[Any, Any]](reports.items()):
    explained_nuclear_norm = report["explained_nuclear_norm"]
    indices_considered = np.arange(1, len(explained_nuclear_norm) + 1)
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key)
    plt.plot(indices_considered, explained_nuclear_norm, marker="o", color=colors[idx], label=pretty_key, alpha=0.7)
plt.xlabel("Top singular values", fontweight="bold")
plt.xticks(indices_considered)
plt.title("Explained Nuclear Norm", fontweight="bold")
plt.tight_layout()
plt.legend(frameon=True, loc="lower right")

In [ ]:
# Plot 2: Entropy vs alignment (if available)
plt.figure(figsize=(10, 6))
has_per_query = False
for idx, (key, report) in enumerate(reports.items()):
    if report["per_query"] is not None:
        has_per_query = True
        entropies = report["per_query"]["entropies"]
        alignment_scores = report["per_query"]["alignment_scores"]
        plt.scatter(alignment_scores, entropies, s=10, color=colors[idx], label=key, alpha=0.6)

if has_per_query:
    plt.xlabel("Alignment score", fontweight="bold")
    plt.ylabel("Softmax entropy", fontweight="bold")
    plt.title("Row entropy vs alignment (all heads)", fontweight="bold")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    plt.close()
    print("No per_query data available for entropy vs alignment plot")

In [ ]:
# # Plot 3: Principal angles
# plt.figure(figsize=(5, 5))
# for idx, (key, report) in enumerate(reports.items()):
#     angles_rad = report["principal_angles_radians"]
#     if angles_rad.size > 0:
#         plt.plot(np.degrees(angles_rad), marker="o", color=colors[idx], label=key, alpha=0.7)

# plt.xlabel("Index", fontweight="bold")
# plt.ylabel("Angle (degrees)", fontweight="bold")
# plt.title("Principal angles (all heads)", fontweight="bold")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
reports.keys()

In [ ]:
# Helper function to create combined corner plot
def plot_combined_corner(
    U_vectors: np.ndarray,
    V_vectors: np.ndarray,
    title: str,
    num_svs: int = 6,
    xlim: tuple[float, float] | None = None,
    ylim: tuple[float, float] | None = None,
):
    fig, axes = plt.subplots(num_svs, num_svs, figsize=(3 * num_svs, 3 * num_svs))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(num_svs):
        for j in range(num_svs):
            ax = axes[i, j]
            if i == j:
                # Diagonal: histograms for both U and V
                combined_data = np.concatenate([U_vectors[:, i], V_vectors[:, i]])
                bins = np.linspace(combined_data.min(), combined_data.max(), 30 + 1)
                ax.hist(U_vectors[:, i], bins=bins, color="tab:blue", alpha=0.5, label="U (Left SVs)")
                ax.hist(V_vectors[:, i], bins=bins, color="tab:red", alpha=0.5, label="V (Right SVs)")
                ax.set_title(f"SV {i + 1}", fontweight="bold")
                ax.legend()
            elif i > j:
                # Lower triangle: left singular vectors (U)
                ax.hist2d(U_vectors[:, j], U_vectors[:, i], bins=30, cmap="Blues")
                if xlim is not None:
                    ax.set_xlim(xlim)
                if ylim is not None:
                    ax.set_ylim(ylim)
            else:
                # Upper triangle: right singular vectors (V)
                ax.hist2d(V_vectors[:, j], V_vectors[:, i], bins=30, cmap="Oranges")
                if xlim is not None:
                    ax.set_xlim(xlim)
                if ylim is not None:
                    ax.set_ylim(ylim)

    # plt.subplots_adjust(top=0.92)
    plt.tight_layout()
    plt.show()

In [ ]:
# selected_head = "L0_H4_CA"
# selected_report = reports[selected_head]
# num_svs = 6
# # Plot left and right singular vectors
# plot_corner(selected_report["U"][:, :num_svs], "Left Singular Vectors (U)", "tab:blue", "Blues")
# plot_corner(selected_report["V"][:, :num_svs], "Right Singular Vectors (V)", "tab:red", "Oranges")


In [ ]:
# selected_head = "L4_H1_CA"
# selected_report = reports[selected_head]
# num_svs = 6
# # Plot left and right singular vectors
# plot_corner(selected_report["U"][:, :num_svs], "Left Singular Vectors (U)", "tab:blue", "Blues")
# plot_corner(selected_report["V"][:, :num_svs], "Right Singular Vectors (V)", "tab:red", "Oranges")


In [ ]:
selected_head = "L4_H1_CA"
selected_report = reports[selected_head]
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    selected_report["U"][:, :num_svs],
    selected_report["V"][:, :num_svs],
    f"{pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)


In [ ]:
selected_head = "L0_H4_CA"
selected_report = reports[selected_head]
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    selected_report["U"][:, :num_svs],
    selected_report["V"][:, :num_svs],
    f"{pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)


In [ ]:
selected_head = "L9_H5_CA"
selected_report = reports[selected_head]
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    selected_report["U"][:, :num_svs],
    selected_report["V"][:, :num_svs],
    f"{pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)


In [ ]:
selected_head = "L7_H11_CA"
selected_report = reports[selected_head]
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    selected_report["U"][:, :num_svs],
    selected_report["V"][:, :num_svs],
    f"{pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)


In [ ]:
report.keys()

In [ ]:
symmetric_singular_values = {}
symmetric_Us = {}
symmetric_Vs = {}

for idx, (key, report) in enumerate[tuple[Any, Any]](reports.items()):
    symmetric_part = report["symmetric_part"]
    # svd of symmetric part
    U, s, V = np.linalg.svd(symmetric_part)
    symmetric_Us[key] = U
    symmetric_Vs[key] = V
    symmetric_singular_values[key] = s

In [ ]:
k = 30
plt.figure(figsize=(5, 5))
for idx, key in enumerate(symmetric_singular_values.keys()):
    singular_values = symmetric_singular_values[key][:k]
    indices_considered = np.arange(1, len(singular_values) + 1)
    pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", key)
    plt.plot(indices_considered, singular_values, marker="o", color=colors[idx], label=pretty_key, alpha=0.7)
    plt.xlabel("Top-k singular values", fontweight="bold")
    plt.xticks(indices_considered)

plt.title("Singular Values of Symmetric Part", fontweight="bold")
plt.tight_layout()
plt.legend(frameon=True, loc="upper right")

In [ ]:
selected_head = "L4_H1_CA"
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    symmetric_Us[selected_head][:, :num_svs],
    symmetric_Vs[selected_head][:, :num_svs],
    f"Symmetric Part of {pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)


In [ ]:
selected_head = "L0_H4_CA"
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    symmetric_Us[selected_head][:, :num_svs],
    symmetric_Vs[selected_head][:, :num_svs],
    f"Symmetric Part of{pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)


In [ ]:
# Compute (I - U_i V_i^T) U_j

head_i = "L4_H1_CA"
head_j = "L0_H4_CA"

U_i = symmetric_Us[head_i]
U_j = symmetric_Us[head_j]
V_i = symmetric_Vs[head_i]

# U_i = reports[head_i]["U"]
# V_i = reports[head_i]["V"]
# U_j = reports[head_j]["U"]

# Compute (I - U_i U_i^T) U_j

mat = np.eye(U_i.shape[0]) - U_i @ U_i.T


In [ ]:
mat.shape

In [ ]:
# svd of mat
U_mat, s_mat, V_mat = np.linalg.svd(mat)

In [ ]:
selected_head = "L0_H4_CA"
num_svs = 6
pretty_key = re.sub(r"L(\d+)_H(\d+)_(\w+)", r"L\1 H\2", selected_head)

# Plot combined left and right singular vectors
plot_combined_corner(
    U_mat[:, :num_svs],
    V_mat[:, :num_svs],
    f"{pretty_key}: Singular Subspaces: U (Output Space) and V (Input Space)",
)
